# Stage 3 Exploration — Relational Join & Personalization Engine

Before this becomes `stage3_join_and_score.py`, let's actually **build the
logic here first** — load real data, do the naive thing, look at what's
wrong with it, fix it, and only lock it into a module once it feels right.

**What Stage 3 needs to do:**
1. Take the 20 candidate products Stage 2A already ranked
2. Join them against `inventory_pricing.csv` — each product has multiple SKUs (sizes)
3. Filter: drop out-of-stock SKUs, drop SKUs that don't match the customer's size
4. Compute a **composite** relevance score per SKU — not just the text-match
   score from Stage 2A, but blended with category fit, personalization, and
   possibly price/discount signals
5. Return the final ranked SKU list

We already know from testing Stage 2 that pure text relevance has a real
flaw: a product literally *named* "Coastal Shorts" can outrank a correctly
-categorized "Alpine Parka" for a wedding-in-October-on-the-coast query. Part
of what we're doing here is **fixing that**, with real data in front of us
instead of guessing at weights in the abstract.

## 0. Setup — load real data, get a real candidate list from Stage 2

We reuse Stage 1/2's already-tested code to get a real, real `candidate_products`
list — no point re-inventing that here. This notebook starts from *their*
output.

In [1]:
import json
import pandas as pd

from search_agent.utils.data_loaders import load_products, load_customers
from search_agent.utils.text_scoring import rank_products

products = load_products()
customers = load_customers()

print(f"{len(products)} products, {len(customers)} customers loaded")

500 products, 100 customers loaded


In [2]:
products

[{'product_id': 'PRD-1001',
  'name': 'Trailblazer Trench Coat',
  'description': 'Designed for those who want stylish looks without sacrificing performance, this Gore-Tex trench coat is insulated and flexible from morning commutes to weekend adventures. Reviewers consistently praise how tough the construction feels, thanks to adjustable straps. It also features a slim, tapered fit for added versatility.',
  'category': 'Outerwear',
  'material': 'Gore-Tex',
  'occasion': 'Business'},
 {'product_id': 'PRD-1002',
  'name': 'Summit Jacket',
  'description': "This on-trend jacket is crafted from premium Down Fill for a look that works just as well on hiking trails and outdoor excursions. It's built to be cozy and durable, so it holds up to daily wear without losing its shape. The design includes a hidden interior pocket and a taped, seam-sealed construction, giving you both form and function in one piece.",
  'category': 'Outerwear',
  'material': 'Down Fill',
  'occasion': 'Outdoor'},
 {

In [3]:
for prod in products:
    if (prod['product_id'] == 'PRD-1005'):
        print(prod)

{'product_id': 'PRD-1005', 'name': 'Horizon Trench Coat', 'description': "A wardrobe staple, this trench coat pairs elegant styling with a soft feel you'll want to wear all day. Made from Nylon, it's engineered to be hard-wearing season after season. Thoughtful details like a relaxed, roomy fit and reinforced stitching make it ideal while traveling or on the go.", 'category': 'Outerwear', 'material': 'Nylon', 'occasion': 'Travel'}


In [4]:
# Same example from the Stage 2 test drive — deliberately picked because we
# already know it exposes the "Coastal Shorts" ranking problem.
sample_intent = {
    "raw_query": "waterproof jacket for an October coastal wedding",
    "category": "Outerwear",
    "occasion": "Wedding",
    "weather_attribute": "waterproof",
    "material": None, "size": None, "color": None, "max_price": None,
    "keywords": ["coastal", "october"],
}

candidates = rank_products(products, sample_intent, top_k=20)

candidates_df = pd.DataFrame(candidates)
candidates_df[["product_id", "name", "category", "occasion", "_relevance_score"]].head(10)

,product_id,name,category,occasion,_relevance_score
0,PRD-1430,Coastal Shorts,Bottoms,Wedding,9.854905
1,PRD-1293,Alpine Parka,Outerwear,Wedding,8.859477
2,PRD-1086,Coastal Chinos,Bottoms,Formal,6.971030
3,PRD-1119,Coastal Shorts,Bottoms,Wedding,6.572122
4,PRD-1234,Coastal Hoodie,Tops,Wedding,6.572122
5,PRD-1298,Coastal Sandal,Footwear,Wedding,6.572122
6,PRD-1131,Coastal Windbreaker,Outerwear,Activewear,6.381067
7,PRD-1404,Coastal Windbreaker,Outerwear,Outdoor,6.381067
8,PRD-1426,Coastal Trench Coat,Outerwear,Casual,6.381067
9,PRD-1227,Vantage Polo,Tops,Wedding,6.166657


There it is, reproduced with real numbers: **"Coastal Shorts" is #1**, ahead
of "Alpine Parka" which is actually Outerwear/Wedding — the correct
category *and* occasion. Pure text-match relevance isn't enough on its own.
Keep this DataFrame around — we'll fix the ordering further down.

In [5]:
sample_customer = customers["CUST-001"]
print(json.dumps(sample_customer, indent=1))

{
 "customer_id": "CUST-001",
 "name": "Allison Hill",
 "size_profile": {
  "tops": "S",
  "bottoms": "30",
  "shoes": "7"
 },
 "style_preferences": [
  "classic",
  "athleisure",
  "durable",
  "vintage"
 ],
 "purchase_history": [
  "PRD-1005"
 ]
}


## 1. The Join — try the naive thing first

`inventory_pricing.csv` has one row **per SKU**, and each product has
multiple SKUs (sizes). Let's just do a plain merge and see what happens to
our 20 candidates.

In [6]:
inventory_df = pd.read_csv("data/inventory_pricing.csv")
print(inventory_df.shape)
inventory_df.head(3)

(1500, 6)


,sku_id,product_id,size,stock_count,base_price,discount_percentage
0,SKU-1001-OneSize,PRD-1112,One Size,26,285.18,0.50
1,SKU-1002-7,PRD-1020,7,3,159.71,0.05
2,SKU-1003-S,PRD-1337,S,18,225.91,0.10


In [7]:
joined = candidates_df.merge(inventory_df, on="product_id", how="inner")
print(f"{len(candidates_df)} candidate products -> {len(joined)} SKU rows after the join")
joined[["product_id", "name", "sku_id", "size", "stock_count", "base_price", "discount_percentage"]].head(10)

20 candidate products -> 52 SKU rows after the join


,product_id,name,sku_id,size,stock_count,base_price,discount_percentage
0,PRD-1430,Coastal Shorts,SKU-1293-30,30,29,70.33,0.4
1,PRD-1430,Coastal Shorts,SKU-1788-40,40,10,110.27,0.0
2,PRD-1430,Coastal Shorts,SKU-2049-36,36,25,31.27,0.0
3,PRD-1430,Coastal Shorts,SKU-2065-36,36,43,177.08,0.0
4,PRD-1293,Alpine Parka,SKU-1292-L,L,14,45.31,0.3
5,PRD-1086,Coastal Chinos,SKU-1278-40,40,38,237.33,0.4
6,PRD-1086,Coastal Chinos,SKU-2067-28,28,0,288.03,0.3
7,PRD-1086,Coastal Chinos,SKU-2076-32,32,10,99.89,0.0
8,PRD-1119,Coastal Shorts,SKU-1355-32,32,1,154.81,0.5
9,PRD-1119,Coastal Shorts,SKU-1570-32,32,9,77.71,0.5


Makes sense — 20 products fanned out to however many SKUs (sizes) each has.
This is exactly why Stage 3 operates on SKUs, not products: "in stock" and
"the right size" are SKU-level facts, not product-level ones. A jacket can
be a perfect match and *still* be unsellable if the only size left is a
Small and the customer wears a Large.

## 2. Filter 1 — drop out-of-stock SKUs

Straightforward: `stock_count > 0`. Let's check how much this actually
removes (we know from the data generator that ~15% of all SKUs are
out-of-stock).

In [8]:
in_stock = joined[joined["stock_count"] > 0].copy()
print(f"{len(joined)} SKUs -> {len(in_stock)} in-stock SKUs "
      f"({(1 - len(in_stock)/len(joined)):.0%} dropped)")

52 SKUs -> 43 in-stock SKUs (17% dropped)


## 3. Filter 2 — size matching

This one needs actual thought, not just a one-liner. A few questions to
work through with real data before writing a filter function:

- Different categories use different size vocab: `Tops`/`Outerwear` use
  letter sizes (S/M/L), `Bottoms` use waist numbers, `Footwear` uses shoe
  sizes, `Accessories` are "One Size".
- The customer's `size_profile` is keyed by `tops` / `bottoms` / `shoes` —
  not by category name directly. We need a mapping.
- What about `Accessories`? There's no accessories entry in `size_profile`
  at all — those should just never be filtered out on size.
- What if there's **no customer_profile at all** (guest session)? We
  shouldn't filter by size — show every size and let the shopper pick.

Let's look at the actual size values in the data before assuming anything.

In [9]:
in_stock.groupby("category")["size"].unique().apply(lambda sizes: sorted(sizes, key=str))

category
Accessories                [One Size]
Bottoms          [30, 32, 36, 40, 42]
Footwear            [10, 11, 7, 8, 9]
Outerwear      [L, M, S, XL, XS, XXL]
Tops                           [L, S]
Name: size, dtype: object

In [10]:
in_stock.head(5)


,product_id,name,description,category,material,occasion,_relevance_score,sku_id,size,stock_count,base_price,discount_percentage
0,PRD-1430,Coastal Shorts,Designed for those who want on-trend looks wit...,Bottoms,Wool Blend,Wedding,9.854905,SKU-1293-30,30,29,70.33,0.4
1,PRD-1430,Coastal Shorts,Designed for those who want on-trend looks wit...,Bottoms,Wool Blend,Wedding,9.854905,SKU-1788-40,40,10,110.27,0.0
2,PRD-1430,Coastal Shorts,Designed for those who want on-trend looks wit...,Bottoms,Wool Blend,Wedding,9.854905,SKU-2049-36,36,25,31.27,0.0
3,PRD-1430,Coastal Shorts,Designed for those who want on-trend looks wit...,Bottoms,Wool Blend,Wedding,9.854905,SKU-2065-36,36,43,177.08,0.0
4,PRD-1293,Alpine Parka,Designed for those who want modern looks witho...,Outerwear,Gore-Tex,Wedding,8.859477,SKU-1292-L,L,14,45.31,0.3


In [11]:
sample_customer

{'customer_id': 'CUST-001',
 'name': 'Allison Hill',
 'size_profile': {'tops': 'S', 'bottoms': '30', 'shoes': '7'},
 'style_preferences': ['classic', 'athleisure', 'durable', 'vintage'],
 'purchase_history': ['PRD-1005']}

In [12]:
# category -> which key in size_profile it should be checked against.
# Accessories intentionally maps to None: never filtered on size.
CATEGORY_TO_SIZE_PROFILE_KEY = {
    "Tops": "tops",
    "Outerwear": "tops",
    "Bottoms": "bottoms",
    "Footwear": "shoes",
    "Accessories": None,
}

def matches_customer_size(row, size_profile):
    """Returns True if this SKU's size is wearable for this customer, or if
    sizing doesn't apply / isn't known (accessories, no profile)."""
    if size_profile is None:
        return True  # guest session: don't filter by size at all
    profile_key = CATEGORY_TO_SIZE_PROFILE_KEY.get(row["category"])
    if profile_key is None:
        return True  # accessories: one-size-fits-all, nothing to check
    preferred_size = size_profile.get(profile_key)
    if preferred_size is None:
        return True  # customer has no preference recorded for this category
    return str(row["size"]) == str(preferred_size)

# quick manual check before applying it to the whole frame
test_rows = in_stock.head(5)
for _, row in test_rows.iterrows():
    print(row["category"], row["size"], "->",
          matches_customer_size(row, sample_customer["size_profile"]))

Bottoms 30 -> True
Bottoms 40 -> False
Bottoms 36 -> False
Bottoms 36 -> False
Outerwear L -> False


In [13]:
sample_customer

{'customer_id': 'CUST-001',
 'name': 'Allison Hill',
 'size_profile': {'tops': 'S', 'bottoms': '30', 'shoes': '7'},
 'style_preferences': ['classic', 'athleisure', 'durable', 'vintage'],
 'purchase_history': ['PRD-1005']}

In [14]:
size_matched = in_stock[
    in_stock.apply(lambda row: matches_customer_size(row, sample_customer["size_profile"]), axis=1)
].copy()

print(f"{len(in_stock)} in-stock SKUs -> {len(size_matched)} matching "
      f"{sample_customer['name']}'s sizes {sample_customer['size_profile']}")
size_matched[["product_id", "name", "category", "size", "stock_count"]].head(10)

43 in-stock SKUs -> 7 matching Allison Hill's sizes {'tops': 'S', 'bottoms': '30', 'shoes': '7'}


,product_id,name,category,size,stock_count
0,PRD-1430,Coastal Shorts,Bottoms,30,29
12,PRD-1234,Coastal Hoodie,Tops,S,37
18,PRD-1426,Coastal Trench Coat,Outerwear,S,47
20,PRD-1426,Coastal Trench Coat,Outerwear,S,38
23,PRD-1331,Alpine Tote Bag,Accessories,One Size,42
24,PRD-1331,Alpine Tote Bag,Accessories,One Size,10
29,PRD-1464,Cascade Trail Runner,Footwear,7,36


That's a big cut — expected, since we're now only keeping the *exact* size
a specific customer wears, out of an already-narrow 20-product candidate
set. Worth remembering for Stage 4: if this comes back empty, the agent's
response should say "in your size" rather than silently returning nothing.

## 4. Composite scoring — fixing "Coastal Shorts"

Before mixing this in with the join/filter results, let's isolate the fix
and prove it on the **unfiltered** 20 candidates first — cleaner to verify
one thing at a time.

**Signal 1 — category/occasion match bonus.** The text scorer treats
"Outerwear" appearing in a product's category field the same as "coastal"
appearing in its name — but a *category match* should count for a lot more
than an incidental keyword hit. Let's add a flat bonus when the product's
category equals the intent's category, and another when occasion matches.

In [26]:
def category_occasion_bonus(row, intent):
    bonus = 0.0
    if intent.get("category") and row["category"] == intent["category"]:
        bonus += 5.0
    if intent.get("occasion") and row["occasion"] == intent["occasion"]:
        bonus += 3.0
    return bonus

candidates_df["_category_bonus"] = candidates_df.apply(
    lambda row: category_occasion_bonus(row, sample_intent), axis=1
)
candidates_df["_score_v2"] = candidates_df["_relevance_score"] + candidates_df["_category_bonus"]

candidates_df.sort_values("_score_v2", ascending=False)[
    ["product_id", "name", "category", "occasion", "_relevance_score", "_category_bonus", "_score_v2"]
].head(8)

,product_id,name,category,occasion,_relevance_score,_category_bonus,_score_v2
1,PRD-1293,Alpine Parka,Outerwear,Wedding,8.859477,8.0,16.859477
0,PRD-1430,Coastal Shorts,Bottoms,Wedding,9.854905,3.0,12.854905
6,PRD-1131,Coastal Windbreaker,Outerwear,Activewear,6.381067,5.0,11.381067
7,PRD-1404,Coastal Windbreaker,Outerwear,Outdoor,6.381067,5.0,11.381067
8,PRD-1426,Coastal Trench Coat,Outerwear,Casual,6.381067,5.0,11.381067
17,PRD-1270,Horizon Bomber Jacket,Outerwear,Travel,5.975602,5.0,10.975602
19,PRD-1436,Trailblazer Trench Coat,Outerwear,Outdoor,5.975602,5.0,10.975602
18,PRD-1375,Cascade Windbreaker,Outerwear,Business,5.975602,5.0,10.975602


That's the fix, on the unfiltered candidate list: "Alpine Parka"
(Outerwear/Wedding, +8.0 bonus) now clearly outranks "Coastal Shorts"
(Bottoms, gets only the +3.0 occasion bonus, no category match). Good — the
bonus values (5.0 / 3.0) are a first guess; we'll leave them as named
constants so they're easy to tune later rather than magic numbers buried in
a formula.

**Now let's see what happens once we re-apply the in-stock + size filters
from sections 2-3.**

In [27]:
size_matched["_category_bonus"] = size_matched.apply(
    lambda row: category_occasion_bonus(row, sample_intent), axis=1
)
size_matched["_score_v2"] = size_matched["_relevance_score"] + size_matched["_category_bonus"]

size_matched.sort_values("_score_v2", ascending=False)[
    ["product_id", "name", "category", "occasion", "_relevance_score", "_category_bonus", "_score_v2"]
].head(8)

,product_id,name,category,occasion,_relevance_score,_category_bonus,_score_v2
0,PRD-1430,Coastal Shorts,Bottoms,Wedding,9.854905,3.0,12.854905
18,PRD-1426,Coastal Trench Coat,Outerwear,Casual,6.381067,5.0,11.381067
20,PRD-1426,Coastal Trench Coat,Outerwear,Casual,6.381067,5.0,11.381067
12,PRD-1234,Coastal Hoodie,Tops,Wedding,6.572122,3.0,9.572122
23,PRD-1331,Alpine Tote Bag,Accessories,Wedding,6.166657,3.0,9.166657
24,PRD-1331,Alpine Tote Bag,Accessories,Wedding,6.166657,3.0,9.166657
29,PRD-1464,Cascade Trail Runner,Footwear,Wedding,6.166657,3.0,9.166657


Wait — **"Alpine Parka" isn't in this list at all**, even though the
category/occasion fix clearly worked one cell ago. Don't gloss over that —
let's find out why, since silently-missing correct answers are exactly the
kind of thing that should get caught in exploration, not in production.

In [18]:
# Was Alpine Parka (PRD-1293) dropped by the join, the stock filter, or the size filter?
alpine_after_join = joined[joined["product_id"] == "PRD-1293"]
print("SKUs for Alpine Parka after the join:")
print(alpine_after_join[["sku_id", "size", "stock_count"]])
print()
print(f"{sample_customer['name']}'s size_profile:", sample_customer["size_profile"])

SKUs for Alpine Parka after the join:
       sku_id size  stock_count
4  SKU-1292-L    L           14

Allison Hill's size_profile: {'tops': 'S', 'bottoms': '30', 'shoes': '7'}


There's the answer: Alpine Parka only comes in size **L**, and this
customer wears a **S** top. It's not a scoring bug — the size filter is
correctly excluding a jacket this specific customer literally cannot wear
in their size. That's a legitimate, important real-world case, not a flaw
to fix here.

**This changes what Stage 4 needs to do**, though: if the single best
category/occasion match gets excluded by an exact-size filter, the final
response shouldn't just silently show worse-matching items — it should
probably say something like *"we don't have the Alpine Parka in your
size, but here's a similar option that does fit."* Noting that as a
requirement for the synthesis node rather than solving it here.

**Signal 2 — personalization**, using the customer's `style_preferences`.
Let's check word overlap between their style tags and the product's
description/category/material/occasion text.

In [31]:
def personalization_bonus(row, style_preferences, weight=1.5):
    if not style_preferences:
        return 0.0
    haystack = f"{row['name']} {row['description']} {row['category']} {row['material']} {row['occasion']}".lower()
    matches = sum(1 for tag in style_preferences if tag.lower() in haystack)
    return matches * weight

size_matched["_personalization_bonus"] = size_matched.apply(
    lambda row: personalization_bonus(row, sample_customer["style_preferences"]), axis=1
)
size_matched["_score_v3"] = size_matched["_score_v2"] + size_matched["_personalization_bonus"]

size_matched.sort_values("_score_v3", ascending=False)[
    ["product_id", "name", "_relevance_score", "_category_bonus", "_personalization_bonus", "_score_v3"]
].head(8)

,product_id,name,_relevance_score,_category_bonus,_personalization_bonus,_score_v3
0,PRD-1430,Coastal Shorts,9.854905,3.0,0.0,12.854905
18,PRD-1426,Coastal Trench Coat,6.381067,5.0,0.0,11.381067
20,PRD-1426,Coastal Trench Coat,6.381067,5.0,0.0,11.381067
12,PRD-1234,Coastal Hoodie,6.572122,3.0,0.0,9.572122
23,PRD-1331,Alpine Tote Bag,6.166657,3.0,0.0,9.166657
24,PRD-1331,Alpine Tote Bag,6.166657,3.0,0.0,9.166657
29,PRD-1464,Cascade Trail Runner,6.166657,3.0,0.0,9.166657


This customer's style tags (check the profile above) barely overlap with a
wedding-jacket search, so this signal isn't doing much for *this* query —
that's fine and expected; it should matter more for style-driven queries
like "something minimalist for the office." Worth testing against a second,
more style-flavored query before we lock in the weight.

**Signal 3 — a small discount nudge.** Not a huge factor, but all else
equal, a discounted item is a reasonable tie-breaker to surface.

In [35]:
DISCOUNT_WEIGHT = 2.0  # discount_percentage is 0.0-0.5, so max contribution is 1.0

size_matched["_discount_bonus"] = size_matched["discount_percentage"] * DISCOUNT_WEIGHT
size_matched["_composite_score"] = size_matched["_score_v3"] + size_matched["_discount_bonus"]

final_ranked = size_matched.sort_values("_composite_score", ascending=False)
final_ranked[
    ["sku_id", "product_id", "name", "size", "base_price", "discount_percentage", "_composite_score"]
].head(8)

,sku_id,product_id,name,size,base_price,discount_percentage,_composite_score
0,SKU-1293-30,PRD-1430,Coastal Shorts,30,70.33,0.40,13.654905
18,SKU-1126-S,PRD-1426,Coastal Trench Coat,S,156.59,0.15,11.681067
20,SKU-1781-S,PRD-1426,Coastal Trench Coat,S,85.68,0.10,11.581067
12,SKU-2115-S,PRD-1234,Coastal Hoodie,S,228.07,0.10,9.772122
29,SKU-2271-7,PRD-1464,Cascade Trail Runner,7,161.69,0.25,9.666657
24,SKU-1610-OneSize,PRD-1331,Alpine Tote Bag,One Size,57.15,0.10,9.366657
23,SKU-1487-OneSize,PRD-1331,Alpine Tote Bag,One Size,115.99,0.00,9.166657


## 5. Sanity check on a second, more style-driven query

Let's stress-test the same pipeline on a query where personalization
*should* matter more, to make sure signal 2 actually pulls its weight and
we're not just tuning to one example.

In [37]:
style_intent = {
    "raw_query": "something minimalist and durable for the office",
    "category": "Tops",
    "occasion": "Business",
    "weather_attribute": None,
    "material": None, "size": None, "color": None, "max_price": None,
    "keywords": ["minimalist", "durable"],
}

style_candidates = rank_products(products, style_intent, top_k=20)
style_df = pd.DataFrame(style_candidates)

style_joined = style_df.merge(inventory_df, on="product_id", how="inner")
style_in_stock = style_joined[style_joined["stock_count"] > 0].copy()

# reuse a customer whose style_preferences actually include "minimalist"
minimalist_customers = {
    cid: c for cid, c in customers.items() if "minimalist" in c["style_preferences"]
}
demo_customer = next(iter(minimalist_customers.values()))
print(demo_customer["name"], demo_customer["style_preferences"], demo_customer["size_profile"])

style_size_matched = style_in_stock[
    style_in_stock.apply(lambda row: matches_customer_size(row, demo_customer["size_profile"]), axis=1)
].copy()

style_size_matched["_category_bonus"] = style_size_matched.apply(
    lambda row: category_occasion_bonus(row, style_intent), axis=1
)
style_size_matched["_personalization_bonus"] = style_size_matched.apply(
    lambda row: personalization_bonus(row, demo_customer["style_preferences"]), axis=1
)
style_size_matched["_discount_bonus"] = style_size_matched["discount_percentage"] * DISCOUNT_WEIGHT
style_size_matched["_composite_score"] = (
    style_size_matched["_relevance_score"]
    + style_size_matched["_category_bonus"]
    + style_size_matched["_personalization_bonus"]
    + style_size_matched["_discount_bonus"]
)

style_size_matched.sort_values("_composite_score", ascending=False)[
    ["sku_id", "product_id", "name", "_relevance_score", "_category_bonus",
     "_personalization_bonus", "_discount_bonus", "_composite_score"]
].head(8)

Daniel Wagner ['techwear', 'casual', 'streetwear', 'lightweight', 'minimalist'] {'tops': 'XXL', 'bottoms': '40', 'shoes': '11'}


,sku_id,product_id,name,_relevance_score,_category_bonus,_personalization_bonus,_discount_bonus,_composite_score
12,SKU-2260-XXL,PRD-1132,Voyager Sweater,8.395198,8.0,1.5,1.0,18.895198
0,SKU-1279-XXL,PRD-1029,Cascade Blouse,8.395198,8.0,1.5,0.3,18.195198
16,SKU-1969-XXL,PRD-1338,Trailblazer Hoodie,8.395198,8.0,0.0,0.5,16.895198
18,SKU-2284-XXL,PRD-1338,Trailblazer Hoodie,8.395198,8.0,0.0,0.5,16.895198
37,SKU-1334-OneSize,PRD-1413,Summit Beanie,5.723885,3.0,1.5,0.8,11.023885
38,SKU-2088-OneSize,PRD-1413,Summit Beanie,5.723885,3.0,1.5,0.4,10.623885
47,SKU-1431-OneSize,PRD-1475,Alpine Tote Bag,5.723885,3.0,0.0,1.0,9.723885
41,SKU-2314-11,PRD-1422,Ridgeline Boot,5.723885,3.0,0.0,0.5,9.223885


Good — for a customer whose style tags actually match the query, the
personalization bonus now visibly separates the top results instead of
sitting at zero for everyone. That's the confirmation we wanted before
locking these weights into a module.

## 6. What we now know, going into the module

Before writing `stage3_join_and_score.py`, here's what this notebook
actually taught us about the logic (this is the point of doing it here
first, rather than guessing at a formula sight-unseen):

1. **The join must fan out to SKU-level rows** — filters and scoring both
   operate per-SKU, not per-product, because in-stock/size are SKU facts.
2. **Size matching needs a category → size_profile-key mapping**
   (`CATEGORY_TO_SIZE_PROFILE_KEY`), and must treat `Accessories` and
   "no customer profile" as **pass-through**, not exclusion.
3. **Raw text relevance alone mis-ranks results** — category/occasion match
   needs an explicit bonus, confirmed by literally watching "Coastal
   Shorts" drop out of first place once it's added.
4. **Personalization only shows its effect on the right kind of query** —
   verified by testing two different intents, not just one.
5. **The composite score is a simple weighted sum** of four named,
   independently-tunable components: text relevance, category/occasion
   bonus, personalization bonus, discount bonus. No exotic math needed.
6. **Empty results are a real, expected case** (narrow candidate set ∩ exact
   size match can legitimately return nothing) — Stage 4's synthesis node
   needs to handle that gracefully, not assume there's always something.

**Open question for you before I write the module:** this notebook used
`pandas` for the join because it's genuinely the right tool for a relational
join + groupby exploration. Stage 2's node code stayed pure-stdlib
(list-of-dicts) by design. For the actual `stage3_*` node, do you want:
- **(a)** pandas-based (matches what we just proved out here, less code to
  translate), or
- **(b)** pure-stdlib dict/list based (matches Stage 2's style, zero pandas
  dependency in the hot path)?

Either is a straightforward translation of the logic above — happy to do
whichever fits how you want the codebase to feel.